 This file is to download DSLD data 

In [1]:
import requests
import pandas as pd 
import json 
from pathlib import Path

In [2]:
BASE_URL = "https://api.ods.od.nih.gov/dsld/v9"

url = f"{BASE_URL}/search-filter"

params = {
    "q": "Magnesium",
    "from": 0,
    "size": 10
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("Status Code:", response.status_code)
print("Response Type:", type(data))
print("Top-Level Keys:", list(data.keys()))
print("Number of Records:", len(data.get("hits", [])))

Status Code: 200
Response Type: <class 'dict'>
Top-Level Keys: ['hits', 'stats']
Number of Records: 10


In [3]:
records = data.get("hits", [])

if records:
    print(json.dumps(records[0], indent=2)[:6000])
else:
    print("No records returned.")

{
  "_index": "dsldnxt_labels",
  "_type": "_doc",
  "_id": "19279",
  "_score": 38.621597,
  "_source": {
    "userGroups": [
      {
        "dailyValueTargetGroupName": "Adults and children 4 or more years of age",
        "langualCodeDescription": "Adults and Children 4 years and above",
        "langualCode": "P0250"
      }
    ],
    "brandName": "Nutra Sport",
    "physicalState": {
      "langualCodeDescription": "Capsule",
      "langualCode": "E0159"
    },
    "entryDate": "2013-05-24",
    "allIngredients": [
      {
        "ingredientGroup": "Magnesium",
        "notes": "Magnesium (Form: from {Magnesium} Amino Acid Chelate, {Magnesium} Citrate, and {Magnesium} Malate)",
        "name": "Magnesium",
        "category": "mineral"
      },
      {
        "ingredientGroup": "Malic Acid",
        "notes": "",
        "name": "Malic Acid",
        "category": "non-nutrient/non-botanical"
      },
      {
        "ingredientGroup": "capsule",
        "notes": "",
        "nam

In [4]:
record_id = data["hits"][0]["_id"]

print(record_id)  # 19279

19279


In [5]:
print(json.dumps(data["stats"], indent=2))

{
  "count": 77199,
  "pct": 0.3594329080920011
}


In [6]:
params = {
    "q": "Magnesium",
    "from": 0,
    "size": 100
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()
records = data["hits"]

print("Records retrieved:", len(records))
print("Total matching records:", data["stats"]["count"])

Records retrieved: 100
Total matching records: 77199


In [7]:
products = []

for record in records:
    source = record["_source"]

    products.append({
        "dsld_id": record["_id"],
        "product_name": source.get("fullName"),
        "brand": source.get("brandName"),
        "ingredient_count": len(source.get("allIngredients") or []),
        "entry_date": source.get("entryDate")
    })

df = pd.DataFrame(products)

display(df.head(10))

,dsld_id,product_name,brand,ingredient_count,entry_date
0,19279,Magnesium,Nutra Sport,5,2013-05-24
1,242165,Quad-Magnesium,VitaMonk,3,2021-03-25
2,183121,Magnesium 3,InterPlexus,7,2018-11-21
3,204235,Magnesium Citrate,Bluebonnet,6,2019-07-24
4,203860,Magnesium Citrate,Bluebonnet,6,2019-07-24
5,47753,Magnesium Citrate,Physio Vites,5,2015-07-24
6,21158,Magnesium Citrate,Vitamin World,4,2013-05-24
7,184238,Magnesium Citrate,Allergy Research Group,5,2019-02-25
8,7841,Magnesium Optimizer,Jarrow Formulas,9,2012-04-26
9,29547,Magnesium Optimizer,Jarrow Formulas,9,2014-01-24


In [8]:
print("Total records:", len(df))

print("Unique DSLD IDs:", df["dsld_id"].nunique())

print("Unique product names:", df["product_name"].nunique())

print("Unique brands:", df["brand"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nIngredient count statistics:")
print(df["ingredient_count"].describe())

Total records: 100
Unique DSLD IDs: 100
Unique product names: 23
Unique brands: 40

Missing values:
dsld_id             0
product_name        0
brand               0
ingredient_count    0
entry_date          0
dtype: int64

Ingredient count statistics:
count    100.000000
mean       6.600000
std        3.253592
min        2.000000
25%        5.000000
50%        5.000000
75%        8.000000
max       18.000000
Name: ingredient_count, dtype: float64


In [9]:
# Duplicate DSLD identifiers
print("Duplicate DSLD IDs:")
print(df[df.duplicated(subset=["dsld_id"], keep=False)])

# Products with identical names and brands
print("\nRepeated product name + brand combinations:")

display(
    df[df.duplicated(
        subset=["product_name", "brand"],
        keep=False
    )].sort_values(["product_name", "brand"])
)

Duplicate DSLD IDs:
Empty DataFrame
Columns: [dsld_id, product_name, brand, ingredient_count, entry_date]
Index: []

Repeated product name + brand combinations:


,dsld_id,product_name,brand,ingredient_count,entry_date
54,244802,Calcium & Magnesium,NOW,8,2021-04-22
55,210414,Calcium & Magnesium,NOW,8,2019-12-22
56,210413,Calcium & Magnesium,NOW,9,2019-12-22
73,14718,Calcium & Magnesium,NOW,7,2012-10-25
74,14717,Calcium & Magnesium,NOW,7,2012-10-25
...,...,...,...,...,...
45,243394,Magnesium Optimizer,Jarrow Formulas,9,2021-03-25
25,68831,Mega Magnesium,Vitamer Laboratories,5,2016-12-23
35,23668,Mega Magnesium,Vitamer Laboratories,5,2013-07-25
63,249889,Perfect Magnesium,Perfect Supplements,3,2021-06-23


In [10]:
from collections import Counter

ingredient_names = []

for record in records:
    ingredients = record["_source"].get("allIngredients") or []

    for ingredient in ingredients:
        name = ingredient.get("name")

        if name:
            ingredient_names.append(name)

ingredient_counts = Counter(ingredient_names)

print("Total ingredient occurrences:", len(ingredient_names))
print("Unique ingredient names:", len(ingredient_counts))

print("\nTop 20 most frequent ingredients:")
for name, count in ingredient_counts.most_common(20):
    print(f"{name}: {count}")

Total ingredient occurrences: 660
Unique ingredient names: 120

Top 20 most frequent ingredients:
Magnesium: 97
Magnesium Stearate: 50
Silicon Dioxide: 35
Microcrystalline Cellulose: 30
Silica: 28
Cellulose: 25
Stearic Acid: 23
Vegetable Cellulose: 18
Vegetable Magnesium Stearate: 18
Hydroxypropyl Methylcellulose: 15
Malic Acid: 12
Total Carbohydrates: 12
Dicalcium Phosphate: 12
Vegetable Glycerin: 12
Vegetarian Capsule: 11
Calcium: 11
Croscarmellose Sodium: 11
Gelatin: 10
Calories: 9
Water: 7


In [11]:
from collections import Counter
import pandas as pd

ingredient_rows = []

for record in records:
    source = record["_source"]

    for ingredient in source.get("allIngredients") or []:
        ingredient_rows.append({
            "dsld_id": record["_id"],
            "ingredient_name": ingredient.get("name"),
            "ingredient_group": ingredient.get("ingredientGroup"),
            "category": ingredient.get("category"),
            "notes": ingredient.get("notes")
        })

ingredients_df = pd.DataFrame(ingredient_rows)

print("Total ingredient records:", len(ingredients_df))

print("\nIngredient categories:")
print(
    ingredients_df["category"]
    .value_counts(dropna=False)
)

print("\nUnique ingredient names by category:")
print(
    ingredients_df.groupby("category", dropna=False)
    ["ingredient_name"].nunique()
)

Total ingredient records: 660

Ingredient categories:
category
mineral                       227
other                         221
fiber                          46
botanical                      37
sugar                          24
fatty acid                     23
non-nutrient/non-botanical     20
fat                            19
vitamin                        14
protein                        13
blend                           7
amino acid                      4
animal part or source           4
bacteria                        1
Name: count, dtype: int64

Unique ingredient names by category:
category
amino acid                     4
animal part or source          2
bacteria                       1
blend                          4
botanical                     18
fat                           10
fatty acid                     1
fiber                          4
mineral                       17
non-nutrient/non-botanical     3
other                         43
protein                  

In [12]:
from collections import Counter

# Count how often each field occurs across product records
field_counts = Counter()

for record in records:
    source = record.get("_source", {})
    field_counts.update(source.keys())

print("Fields present across 100 DSLD records:\n")

for field, count in field_counts.most_common():
    print(f"{field:<35} {count:>3}/100")

Fields present across 100 DSLD records:

userGroups                          100/100
brandName                           100/100
physicalState                       100/100
entryDate                           100/100
allIngredients                      100/100
claims                              100/100
fullName                            100/100
offMarket                           100/100
netContents                         100/100
productType                         100/100
events                              100/100


In [13]:
# Retrieve the complete label for our first product

dsld_id = records[0]["_id"]

label_url = f"{BASE_URL}/label/{dsld_id}"

response = requests.get(label_url, timeout=30)
response.raise_for_status()

label_data = response.json()

print("DSLD ID:", dsld_id)
print("\nAvailable fields:")

for key, value in label_data.items():
    print(f"{key:<35} {type(value).__name__}")

DSLD ID: 19279

Available fields:
src                                 str
id                                  int
nhanesId                            str
bundleName                          str
fullName                            str
brandName                           str
brandIpSymbol                       str
upcSku                              str
productVersionCode                  str
pdf                                 str
thumbnail                           str
servingsPerContainer                str
hasOuterCarton                      bool
percentDvFootnote                   str
labelRelationships                  list
contacts                            list
netContents                         list
physicalState                       dict
servingSizes                        list
targetGroups                        list
productType                         dict
statements                          list
claims                              list
events                              

In [14]:
print("Serving sizes:")
print(json.dumps(label_data.get("servingSizes"), indent=2))

print("\nIngredient rows:")
print(json.dumps(label_data.get("ingredientRows"), indent=2)[:6000])

print("\nLabel statements:")
print(json.dumps(label_data.get("statements"), indent=2)[:3000])

Serving sizes:
[
  {
    "order": 1,
    "minQuantity": 3,
    "maxQuantity": 3,
    "minDailyServings": 1,
    "maxDailyServings": 1,
    "unit": "Capsule(s)",
    "inSFB": true
  }
]

Ingredient rows:
[
  {
    "order": 1,
    "ingredientId": 69771,
    "description": null,
    "notes": "Magnesium (Form: from {Magnesium} Amino Acid Chelate, {Magnesium} Citrate, and {Magnesium} Malate) ",
    "quantity": [
      {
        "servingSizeOrder": 1,
        "servingSizeQuantity": 3,
        "operator": "=",
        "quantity": 400,
        "unit": "mg",
        "dailyValueTargetGroup": [
          {
            "name": "Adults and children 4 or more years of age",
            "operator": "=",
            "percent": 100,
            "footnote": null
          }
        ],
        "servingSizeUnit": "Capsule(s)"
      }
    ],
    "nestedRows": [],
    "name": "Magnesium",
    "category": "mineral",
    "ingredientGroup": "Magnesium",
    "uniiCode": null,
    "alternateNames": [],
    "form